In [73]:
data = [
    # From AIMA Figure 18.3 (attributes renamed a bit for code-friendliness)
    {"Alt": True,  "Bar": False, "FriSat": False, "Hungry": True,  "Patrons": "Some", "Price": "$$$", "Rain": False, "Res": True,  "Type": "French", "Wait": "0-10",  "WillWait": "Yes"},
    {"Alt": True,  "Bar": False, "FriSat": False, "Hungry": True,  "Patrons": "Full", "Price": "$",   "Rain": False, "Res": False, "Type": "Thai",   "Wait": "30-60","WillWait": "No"},
    {"Alt": False, "Bar": True,  "FriSat": False, "Hungry": False, "Patrons": "Some", "Price": "$",   "Rain": False, "Res": False, "Type": "Burger", "Wait": "0-10",  "WillWait": "Yes"},
    {"Alt": True,  "Bar": False, "FriSat": True,  "Hungry": True,  "Patrons": "Full", "Price": "$",   "Rain": True,  "Res": False, "Type": "Thai",   "Wait": "10-30","WillWait": "Yes"},
    {"Alt": True,  "Bar": False, "FriSat": True,  "Hungry": False, "Patrons": "Full", "Price": "$$$", "Rain": False, "Res": True,  "Type": "French", "Wait": ">60",   "WillWait": "No"},
    {"Alt": False, "Bar": True,  "FriSat": False, "Hungry": True,  "Patrons": "Some", "Price": "$$",  "Rain": True,  "Res": True,  "Type": "Italian","Wait": "0-10",  "WillWait": "Yes"},
    {"Alt": False, "Bar": True,  "FriSat": False, "Hungry": False, "Patrons": "None", "Price": "$",   "Rain": True,  "Res": False, "Type": "Burger", "Wait": "0-10",  "WillWait": "No"},
    {"Alt": False, "Bar": False, "FriSat": False, "Hungry": True,  "Patrons": "Some", "Price": "$$",  "Rain": True,  "Res": True,  "Type": "Thai",   "Wait": "0-10",  "WillWait": "Yes"},
    {"Alt": False, "Bar": True,  "FriSat": True,  "Hungry": False, "Patrons": "Full", "Price": "$",   "Rain": True,  "Res": False, "Type": "Burger", "Wait": ">60",   "WillWait": "No"},
    {"Alt": True,  "Bar": True,  "FriSat": True,  "Hungry": True,  "Patrons": "Full", "Price": "$$$", "Rain": False, "Res": True,  "Type": "Italian","Wait": "10-30","WillWait": "No"},
    {"Alt": False, "Bar": False, "FriSat": False, "Hungry": False, "Patrons": "None", "Price": "$",   "Rain": False, "Res": False, "Type": "Thai",   "Wait": "0-10",  "WillWait": "No"},
    {"Alt": True,  "Bar": True,  "FriSat": True,  "Hungry": True,  "Patrons": "Full", "Price": "$",   "Rain": False, "Res": False, "Type": "Burger", "Wait": "30-60","WillWait": "Yes"},
]

feature_names = ["Alt","Bar","FriSat","Hungry","Patrons","Price","Rain","Res","Type","Wait"]
all_feature_levels = {
    "Alt": [True, False, None],
    "Bar": [True, False, None],
    "FriSat": [True, False, None],
    "Hungry": [True, False, None],
    "Patrons": ["None", "Some", "Full", "Many"],      # "Many" doesn't appear in the data
    "Price": ["$", "$$", "$$$", "Free"],              # "Free" doesn't appear in the data
    "Rain": [True, False, None],
    "Res": [True, False, None],
    "Type": ["French", "Thai", "Burger", "Italian", "Mexican"], # "Mexican" doesn't appear
    "Wait": ["0-10", "10-30", "30-60", ">60", "<0"],  # "<0" doesn't appear
    "WillWait": ["Yes", "No", "Maybe"]                # "Maybe" doesn't appear
}

response_name = "WillWait"

In [74]:
# base cases: stop splitting
# split algorithm:
# - split into groups
# - calculate total entropy at group level
# - compare to entropy in aggregated data
# - do this for all variables
# - choose the variable with the highest entropy reduction
# now create a new tree

# define split algorithm (features_matrix, features)
# for feature_name in feature_names
# split based on that feature

# print(value_counts(test_data, "a"))

In [75]:
from collections import Counter

def value_counts(data, count_variable_name):
    return Counter(row[count_variable_name] for row in data)

print(value_counts(data, "Alt"))

Counter({True: 6, False: 6})


In [76]:
from collections import defaultdict

def group_by(data, feature_name, feature_levels):
    grouped_data = defaultdict(list)
    for row in data:
        feature_value = row[feature_name]
        for feature_level in feature_levels:
            if feature_value == feature_level:
                grouped_data[feature_level].append(row)
    for feature_level in feature_levels:
        grouped_data.setdefault(feature_level, [])
    return grouped_data

feature_name="Alt"
feature_levels = all_feature_levels["Alt"]
data_grouped = group_by(data=data, feature_name=feature_name, feature_levels=feature_levels)

# print(data_grouped)
print(data_grouped.keys())

dict_keys([True, False, None])


In [77]:
import math

def neg_expected_log_prob(p):
    if p in {0,1}:
        return 0
    return - p * math.log(p)

def calculate_entropy(p):
    """binomial entropy"""
    return neg_expected_log_prob(p) + neg_expected_log_prob(1-p)

In [78]:
def fraction_yes(response_counts):
    n_total = sum(response_counts.values())
    if n_total == 0:
        return 0
    return response_counts["Yes"] / n_total

def calculate_information_gain(data, current_feature, response_name, feature_levels):
    data_by_feature = group_by(data, current_feature, feature_levels)

    grouped_response_counts = [value_counts(data=rows, count_variable_name=response_name) for rows in data_by_feature.values()]
    marginal_counts = value_counts(data=data, count_variable_name=response_name)

    p_yes_marginal = fraction_yes(marginal_counts)
    p_yes_for_groups = [fraction_yes(counts) for counts in grouped_response_counts]

    n_rows_by_group = [len(values) for values in data_by_feature.values()]
    total_rows = sum(n_rows_by_group)
    weights_for_groups = [n_rows_group / total_rows if (n_rows_group != 0) else 0 for n_rows_group in n_rows_by_group]

    entropy_marginal = calculate_entropy(p_yes_marginal)
    # bugfix: zip the two lists, do not tuple the lists
    entropy_grouped = sum(weight * calculate_entropy(p_yes) for weight, p_yes in zip(weights_for_groups, p_yes_for_groups))

    return entropy_marginal - entropy_grouped

current_feature = "Alt"
print(calculate_information_gain(data, current_feature, response_name, feature_levels))

0.0


In [79]:
def plurality_value(data, response_name):
    counts = value_counts(data, response_name)
    return counts.most_common(1)[0][0]

def all_same_label(data, response_name):
    return len({row[response_name] for row in data}) <= 1

def best_feature(data, feature_names, response_name, all_feature_levels):
    gains = [(calculate_information_gain(data, f, response_name, all_feature_levels[f]), f)
             for f in feature_names]
    return max(gains, key=lambda t: t[0])[1]

In [ ]:
def build_decision_tree_sketch(data, response_name, all_feature_levels, parent_data=None):
    pass

    # base cases - return VALUES
    # 1. if there's only one category of response left in the data, assign that
    # 2. if there are no features left in the data, assign the most common response value
    # 3. if there's no data left at all, assign the most common response value from parent_data
    # in all these cases, we return just a VALUE, not data

    # prep for the new trees
    # 1. use best_feature to find the name of the best feature
    # 2. use group_by to split the data into subsets by that feature
    # 3. create a new features list without that feature, and remove it from the data
    
    # for each of the grouped datasets:
    # run the algorithm
    # - returns BRANCHES (list of trees or nodes for each group)

    # return node[feature_name, branches]


def build_predict_sketch(datapoint):
    pass

    # also recursive - we go down the tree we just made:
    # base case - if it's a VALUE, we return that
    # otherwise - if its BRANCH, we choose ONE path


In [81]:
def build_decision_tree(data, feature_names, response_name, all_feature_levels, parent_data=None):
    if not data:
        return {"value": plurality_value(parent_data, response_name)}
    if all_same_label(data, response_name):
        return {"value": data[0][response_name]}
    if not feature_names:
        return {"value": plurality_value(data, response_name)}

    A = best_feature(data, feature_names, response_name, all_feature_levels)
    node_default = plurality_value(data, response_name)
    node = {"feature": A, "branches": {}, "default": node_default}

    levels = all_feature_levels[A]
    splits = group_by(data, A, levels)
    remaining = [f for f in feature_names if f != A]

    for v in levels:
        subset = splits[v]
        if not subset:
            node["branches"][v] = {"value": node_default}
        else:
            node["branches"][v] = build_decision_tree(subset, remaining, response_name, all_feature_levels, data)
    return node

def predict(tree, row):
    if "value" in tree:
        return tree["value"]
    v = row.get(tree["feature"])
    child = tree["branches"].get(v)
    if child is None:
        return tree["default"]
    return predict(child, row)

In [83]:
tree = build_decision_tree(data, feature_names, response_name, all_feature_levels)
pred0 = predict(tree, data[0])

tree

{'feature': 'Patrons',
 'branches': {'None': {'value': 'No'},
  'Some': {'value': 'Yes'},
  'Full': {'feature': 'Hungry',
   'branches': {True: {'feature': 'Type',
     'branches': {'French': {'value': 'No'},
      'Thai': {'feature': 'FriSat',
       'branches': {True: {'value': 'Yes'},
        False: {'value': 'No'},
        None: {'value': 'No'}},
       'default': 'No'},
      'Burger': {'value': 'Yes'},
      'Italian': {'value': 'No'},
      'Mexican': {'value': 'No'}},
     'default': 'No'},
    False: {'value': 'No'},
    None: {'value': 'No'}},
   'default': 'No'},
  'Many': {'value': 'Yes'}},
 'default': 'Yes'}